# 🎯 AI Hallucination Detection System

**Project Goal:** Build an ML model that detects when AI responses contain hallucinations and calculate a hallucination score.

**What are AI Hallucinations?**
- AI generates false or made-up information
- Confident statements without factual basis
- Contradictory or inconsistent responses
- Overly specific details that seem fabricated

**Our Approach:**
1. Create a training dataset with examples
2. Extract features from text (confidence patterns, specificity, consistency)
3. Train a classifier model
4. Build an interface to test any Q&A pair

In [13]:
# Install required libraries (run this first!)
# Using %pip which works better in VS Code notebooks
%pip install scikit-learn pandas numpy textblob pymongo -q

Note: you may need to restart the kernel to use updated packages.


In [14]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import re
from textblob import TextBlob
from pymongo import MongoClient
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 🗄️ MongoDB Setup

Connect to MongoDB to store training data and detection results.

In [15]:
# MongoDB Connection Setup
# Update the connection string with your MongoDB details
MONGO_URI = "mongodb://localhost:27017/"  # Change this if using MongoDB Atlas or different host
DATABASE_NAME = "Mirage_Mind"  # Your database name

try:
    # Connect to MongoDB
    client = MongoClient(MONGO_URI)
    db = client[DATABASE_NAME]
    
    # Connect to your existing collections
    datas = db['datas']           # Collection 1
    datas2 = db['datas2']         # Collection 2
    datas3 = db['datas3']         # Collection 3
    
    # Also create references for the notebook's default collections
    training_data_collection = datas      # Using datas for training data
    detection_results_collection = datas2  # Using datas2 for detection results
    
    # Test connection
    client.server_info()
    print("✅ Successfully connected to MongoDB!")
    print(f"📁 Database: {DATABASE_NAME}")
    print(f"📊 Your Collections: datas, datas2, datas3")
    print(f"\n💡 Collection assignments:")
    print(f"   • datas  → training data")
    print(f"   • datas2 → detection results")
    print(f"   • datas3 → available for custom use")
    
    # Show what's in your collections
    print(f"\n📈 Collection stats:")
    print(f"   • datas:  {datas.count_documents({})} documents")
    print(f"   • datas2: {datas2.count_documents({})} documents")
    print(f"   • datas3: {datas3.count_documents({})} documents")
    
except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    print("\n💡 Make sure MongoDB is running on your system!")
    print("   Download from: https://www.mongodb.com/try/download/community")

✅ Successfully connected to MongoDB!
📁 Database: Mirage_Mind
📊 Your Collections: datas, datas2, datas3

💡 Collection assignments:
   • datas  → training data
   • datas2 → detection results
   • datas3 → available for custom use

📈 Collection stats:
   • datas:  500 documents
   • datas2: 1007 documents
   • datas3: 5000 documents


## 📊 Step 1: Load Raw Data from MongoDB

Load data from all MongoDB collections and standardize field names.

In [16]:
# 💾 Load raw data from all MongoDB collections
print("📥 Loading raw data from MongoDB collections...\n")

try:
    # Load data from all three collections
    datas_records = list(datas.find({}))
    datas2_records = list(datas2.find({}))
    datas3_records = list(datas3.find({}))
    
    print(f"📊 Records found:")
    print(f"   • datas:  {len(datas_records)} documents")
    print(f"   • datas2: {len(datas2_records)} documents")
    print(f"   • datas3: {len(datas3_records)} documents")
    
    # Combine all records
    all_records = datas_records + datas2_records + datas3_records
    
    if all_records:
        # Clean records - standardize field names
        cleaned_records = []
        for record in all_records:
            record.pop('_id', None)
            record.pop('created_at', None)
            record.pop('data_type', None)
            record.pop('source', None)
            record.pop('id', None)
            record.pop('context', None)
            record.pop('domain', None)
            record.pop('model_confidence', None)
            record.pop('source_type', None)
            
            # Standardize field names for different collections
            # Handle datas3 collection format
            if 'user_query' in record:
                record['question'] = record.pop('user_query')
            if 'model_response' in record:
                record['answer'] = record.pop('model_response')
            if 'hallucination_label' in record:
                record['label'] = record.pop('hallucination_label')
            
            # Handle datas/datas2 format
            if 'ai_answer' in record and 'answer' not in record:
                record['answer'] = record.pop('ai_answer')
            
            # Now convert labels to integers
            if 'label' in record:
                label = record['label']
                # Convert string labels to integers
                if isinstance(label, str):
                    if label.lower() in ['hallucinated', 'hallucination', '1']:
                        record['is_hallucination'] = 1
                    elif label.lower() in ['factual', 'accurate', 'not_hallucinated', '0']:
                        record['is_hallucination'] = 0
                    else:
                        continue  # Skip unknown labels
                elif isinstance(label, (int, float)):
                    record['is_hallucination'] = int(label)
                record.pop('label', None)
            
            # Only keep records with required fields
            if 'question' in record and 'answer' in record and 'is_hallucination' in record:
                cleaned_records.append({
                    'question': record['question'],
                    'answer': record['answer'],
                    'is_hallucination': record['is_hallucination']
                })
        
        print(f"\n✅ Successfully processed {len(cleaned_records)} records")
        
        if len(cleaned_records) == 0:
            print("\n⚠️ WARNING: No records matched the expected structure!")
            raise ValueError("No valid records found")
        
        # Create pandas DataFrame
        df = pd.DataFrame(cleaned_records)
        
        print(f"\n📊 DataFrame Info:")
        print(f"   Columns: {list(df.columns)}")
        print(f"   Shape: {df.shape}")
        
        # Show label distribution
        print(f"\n📈 Label Distribution:")
        print(f"   Non-hallucinated (0): {(df['is_hallucination']==0).sum()}")
        print(f"   Hallucinated (1): {(df['is_hallucination']==1).sum()}")
        
        # Display sample data
        print(f"\n🔍 Sample raw data:")
        display(df.head(5))
        
        print(f"\n💡 Raw data loaded successfully! Ready for cleaning and preprocessing.")
        
    else:
        print("\n⚠️ No records found in any collection!")
        raise ValueError("No records found")
        
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\n⚠️ Could not load data from MongoDB.")
    print("Creating sample synthetic dataset for demonstration...")
    
    # Fallback: Create small synthetic dataset
    data = {
        'question': [
            "What is 2+2?", 
            "Who invented time travel?",
            "What is the capital of France?",
            "Dragons went extinct in what year?"
        ],
        'answer': [
            "2+2 equals 4.", 
            "Dr. James Henderson invented time travel in 1987.",
            "Paris is the capital of France.",
            "Dragons went extinct in 1456 during the Great Dragon War."
        ],
        'is_hallucination': [0, 1, 0, 1]
    }
    df = pd.DataFrame(data)
    train_df, test_df = train_test_split(df, test_size=0.5, random_state=42)
    
    print(f"\n✅ Using {len(df)} synthetic examples for demonstration")
    display(df)

📥 Loading raw data from MongoDB collections...

📊 Records found:
   • datas:  500 documents
   • datas2: 1007 documents
   • datas3: 5000 documents

✅ Successfully processed 6500 records

📊 DataFrame Info:
   Columns: ['question', 'answer', 'is_hallucination']
   Shape: (6500, 3)

📈 Label Distribution:
   Non-hallucinated (0): 3238
   Hallucinated (1): 3262

🔍 Sample raw data:


,question,answer,is_hallucination
0,Who discovered gravity?,Albert Einstein is associated with this fact.,1
1,Who wrote Hamlet?,William Shakespeare is associated with this fact.,0
2,Who wrote Hamlet?,Charles Dickens is associated with this fact.,1
3,Who wrote Hamlet?,William Shakespeare is associated with this fact.,0
4,Who discovered gravity?,Albert Einstein is associated with this fact.,1



💡 Raw data loaded successfully! Ready for cleaning and preprocessing.


## 🧹 Step 2: Data Cleaning and Preprocessing

Clean the data, handle missing values, remove duplicates, and create train-test split.

**Cleaning Steps:**
- Check for missing values
- Remove duplicates
- Clean text (normalize whitespace)
- Filter extreme answer lengths
- Verify label distribution
- Create train-test split (70-30)

In [17]:
# 🧹 Data Cleaning and Preprocessing
print("🧹 Starting data cleaning and preprocessing...\n")

# Check if df exists (must run Cell 7 first!)
try:
    df
except NameError:
    print("❌ ERROR: 'df' is not defined!")
    print("\n💡 You must run Cell 7 (Step 1: Load Raw Data) FIRST before running this cell.")
    print("   The data cleaning step needs the 'df' DataFrame that Cell 7 creates.")
    print("\n📝 To fix:")
    print("   1. Scroll up to Cell 7 (starts with '📥 Loading raw data...')")
    print("   2. Click on that cell")
    print("   3. Press Shift+Enter to run it")
    print("   4. Wait for it to finish loading your data")
    print("   5. Then come back and run this cell")
    raise

# 1. Check for missing values
print("📋 Step 1: Checking for missing values...")
missing_counts = df.isnull().sum()
print(f"   Missing values per column:")
for col, count in missing_counts.items():
    print(f"      {col}: {count}")

# Remove rows with missing values
if df.isnull().any().any():
    before_count = len(df)
    df = df.dropna()
    print(f"   ✅ Removed {before_count - len(df)} rows with missing values")
else:
    print(f"   ✅ No missing values found")

# 2. Remove duplicates
print(f"\n📋 Step 2: Removing duplicate entries...")
before_count = len(df)
df = df.drop_duplicates(subset=['question', 'answer'], keep='first')
removed = before_count - len(df)
print(f"   ✅ Removed {removed} duplicate entries")
print(f"   Dataset size: {len(df)} unique samples")

# 3. Text cleaning
print(f"\n📋 Step 3: Cleaning text data...")

def clean_text(text):
    """Clean text by removing extra whitespace and normalizing"""
    if not isinstance(text, str):
        return str(text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    # Remove leading/trailing whitespace
    text = text.strip()
    return text

# Apply text cleaning
df['question'] = df['question'].apply(clean_text)
df['answer'] = df['answer'].apply(clean_text)

print(f"   ✅ Text cleaned (whitespace normalized)")

# 4. Remove very short or very long answers (likely noise)
print(f"\n📋 Step 4: Filtering answer lengths...")
before_count = len(df)
df = df[(df['answer'].str.len() >= 10) & (df['answer'].str.len() <= 1000)]
removed = before_count - len(df)
print(f"   ✅ Removed {removed} samples with extreme answer lengths")
print(f"   Valid range: 10-1000 characters")

# 5. Check label distribution
print(f"\n📋 Step 5: Verifying label distribution...")
label_counts = df['is_hallucination'].value_counts().sort_index()
print(f"   Non-hallucinated (0): {label_counts.get(0, 0)} ({label_counts.get(0, 0)/len(df)*100:.1f}%)")
print(f"   Hallucinated (1): {label_counts.get(1, 0)} ({label_counts.get(1, 0)/len(df)*100:.1f}%)")

# Check for class imbalance
if len(label_counts) == 2:
    imbalance_ratio = max(label_counts) / min(label_counts)
    if imbalance_ratio > 3:
        print(f"   ⚠️  Warning: Class imbalance detected (ratio: {imbalance_ratio:.2f}:1)")
    else:
        print(f"   ✅ Labels are reasonably balanced (ratio: {imbalance_ratio:.2f}:1)")

# 6. Train-Test Split
print(f"\n📋 Step 6: Creating Train-Test Split (70-30)...")
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, 
    test_size=0.3, 
    random_state=42, 
    stratify=df['is_hallucination']
)

print(f"   ✅ Training set: {len(train_df)} samples")
print(f"      - Non-hallucinated: {(train_df['is_hallucination']==0).sum()}")
print(f"      - Hallucinated: {(train_df['is_hallucination']==1).sum()}")

print(f"   ✅ Testing set: {len(test_df)} samples")
print(f"      - Non-hallucinated: {(test_df['is_hallucination']==0).sum()}")
print(f"      - Hallucinated: {(test_df['is_hallucination']==1).sum()}")

# 7. Summary
print(f"\n" + "="*60)
print("✅ DATA CLEANING COMPLETE")
print("="*60)
print(f"📊 Final Dataset Stats:")
print(f"   Total samples: {len(df)}")
print(f"   Training samples: {len(train_df)}")
print(f"   Testing samples: {len(test_df)}")
print(f"   Features: question, answer, is_hallucination")
print(f"\n💡 Data is clean and ready for feature engineering!")
print("="*60)

# Display cleaned sample
print(f"\n🔍 Sample of cleaned data:")
display(train_df.head(3))

🧹 Starting data cleaning and preprocessing...

📋 Step 1: Checking for missing values...
   Missing values per column:
      question: 0
      answer: 0
      is_hallucination: 0
   ✅ No missing values found

📋 Step 2: Removing duplicate entries...
   ✅ Removed 6378 duplicate entries
   Dataset size: 122 unique samples

📋 Step 3: Cleaning text data...
   ✅ Text cleaned (whitespace normalized)

📋 Step 4: Filtering answer lengths...
   ✅ Removed 0 samples with extreme answer lengths
   Valid range: 10-1000 characters

📋 Step 5: Verifying label distribution...
   Non-hallucinated (0): 61 (50.0%)
   Hallucinated (1): 61 (50.0%)
   ✅ Labels are reasonably balanced (ratio: 1.00:1)

📋 Step 6: Creating Train-Test Split (70-30)...
   ✅ Training set: 85 samples
      - Non-hallucinated: 42
      - Hallucinated: 43
   ✅ Testing set: 37 samples
      - Non-hallucinated: 19
      - Hallucinated: 18

✅ DATA CLEANING COMPLETE
📊 Final Dataset Stats:
   Total samples: 122
   Training samples: 85
   Test

,question,answer,is_hallucination
1534,Summarize a well-known topic in finance.,All global stock markets are controlled by a s...,1
1559,Explain an important concept in finance.,Inflation measures the rate at which the gener...,0
1550,State an evidence-based statement about educat...,All universities worldwide follow a single sta...,1


## 🔍 Step 3: Feature Engineering

Extract features that indicate hallucination from the text data.

**Features to extract:**
- Length indicators (character count, word count)
- Specificity indicators (numbers, dates, proper nouns)
- Confidence words (exactly, precisely, etc.)
- Sentiment analysis (polarity, subjectivity)
- Question-answer length ratio
- Punctuation patterns
- Word complexity

In [18]:
def extract_features(question, answer):
    """Extract features from a question-answer pair"""
    features = {}
    
    # 1. Answer length
    features['answer_length'] = len(answer)
    features['answer_word_count'] = len(answer.split())
    
    # 2. Specificity indicators (numbers, dates, specific names)
    features['number_count'] = len(re.findall(r'\d+', answer))
    features['year_mentions'] = len(re.findall(r'\b(19|20)\d{2}\b', answer))
    
    # 3. Confidence words (hallucinations often sound too confident)
    confidence_words = ['exactly', 'precisely', 'definitely', 'certainly', 'absolutely', 
                        'specifically', 'clearly', 'obviously', 'undoubtedly']
    features['confidence_words'] = sum(1 for word in confidence_words if word in answer.lower())
    
    # 4. Proper nouns (fake names/places)
    words = answer.split()
    features['capitalized_words'] = sum(1 for word in words if word and word[0].isupper() and word.lower() not in ['the', 'a', 'an'])
    
    # 5. Sentiment analysis
    blob = TextBlob(answer)
    features['sentiment_polarity'] = blob.sentiment.polarity
    features['sentiment_subjectivity'] = blob.sentiment.subjectivity
    
    # 6. Question-answer length ratio
    features['qa_length_ratio'] = len(answer) / max(len(question), 1)
    
    # 7. Unusual punctuation (excessive periods, commas)
    features['comma_count'] = answer.count(',')
    features['period_count'] = answer.count('.')
    
    # 8. Complexity
    features['avg_word_length'] = np.mean([len(word) for word in answer.split()]) if answer.split() else 0
    
    return features

print("✅ Feature extraction function defined!")
print(f"\n📊 Features that will be extracted:")
print(f"   1. answer_length - Total characters in answer")
print(f"   2. answer_word_count - Number of words")
print(f"   3. number_count - Count of numbers in answer")
print(f"   4. year_mentions - References to years (19xx, 20xx)")
print(f"   5. confidence_words - Words like 'exactly', 'precisely'")
print(f"   6. capitalized_words - Proper nouns count")
print(f"   7. sentiment_polarity - Emotional tone (-1 to 1)")
print(f"   8. sentiment_subjectivity - Subjective vs objective (0 to 1)")
print(f"   9. qa_length_ratio - Answer length / Question length")
print(f"   10. comma_count - Number of commas")
print(f"   11. period_count - Number of periods")
print(f"   12. avg_word_length - Average word length")

✅ Feature extraction function defined!

📊 Features that will be extracted:
   1. answer_length - Total characters in answer
   2. answer_word_count - Number of words
   3. number_count - Count of numbers in answer
   4. year_mentions - References to years (19xx, 20xx)
   5. confidence_words - Words like 'exactly', 'precisely'
   6. capitalized_words - Proper nouns count
   7. sentiment_polarity - Emotional tone (-1 to 1)
   8. sentiment_subjectivity - Subjective vs objective (0 to 1)
   9. qa_length_ratio - Answer length / Question length
   10. comma_count - Number of commas
   11. period_count - Number of periods
   12. avg_word_length - Average word length


## 🤖 Step 4: Train Machine Learning Model

Train Random Forest Classifier to detect hallucinations based on extracted features.

In [19]:
# 🤖 Train Machine Learning Model using the train-test split from Step 2

print("⚙️ Extracting features from training data...")

# Extract features from training set
train_features = []
for idx, row in train_df.iterrows():
    features = extract_features(row['question'], row['answer'])
    train_features.append(features)

X_train = pd.DataFrame(train_features)
y_train = train_df['is_hallucination'].values

print(f"✅ Training features extracted: {X_train.shape}")

# Extract features from test set
print("⚙️ Extracting features from test data...")
test_features = []
for idx, row in test_df.iterrows():
    features = extract_features(row['question'], row['answer'])
    test_features.append(features)

X_test = pd.DataFrame(test_features)
y_test = test_df['is_hallucination'].values

print(f"✅ Test features extracted: {X_test.shape}")

# Train Random Forest Classifier
print(f"\n🤖 Training Random Forest Classifier...")
print(f"   Model: RandomForestClassifier")
print(f"   Parameters: n_estimators=100, max_depth=10")

model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
model.fit(X_train, y_train)

print(f"✅ Model trained successfully!")

# Make predictions on test set
print(f"\n🔮 Making predictions on test set...")
y_pred = model.predict(X_test)

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
print(f"\n🎯 Model Performance:")
print(f"   Accuracy: {accuracy*100:.2f}%")

print(f"\n📊 Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Hallucination', 'Hallucination']))

# Confusion Matrix
print(f"📉 Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"   True Negatives:  {cm[0][0]}")
print(f"   False Positives: {cm[0][1]}")
print(f"   False Negatives: {cm[1][0]}")
print(f"   True Positives:  {cm[1][1]}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🔥 Top 5 Important Features for Detection:")
print(feature_importance.head())

print("\n✅ Model is ready to detect hallucinations!")

⚙️ Extracting features from training data...
✅ Training features extracted: (85, 12)
⚙️ Extracting features from test data...
✅ Test features extracted: (37, 12)

🤖 Training Random Forest Classifier...
   Model: RandomForestClassifier
   Parameters: n_estimators=100, max_depth=10
✅ Model trained successfully!

🔮 Making predictions on test set...

🎯 Model Performance:
   Accuracy: 86.49%

📊 Detailed Classification Report:
                  precision    recall  f1-score   support

No Hallucination       0.85      0.89      0.87        19
   Hallucination       0.88      0.83      0.86        18

        accuracy                           0.86        37
       macro avg       0.87      0.86      0.86        37
    weighted avg       0.87      0.86      0.86        37

📉 Confusion Matrix:
   True Negatives:  17
   False Positives: 2
   False Negatives: 3
   True Positives:  15

🔥 Top 5 Important Features for Detection:
              feature  importance
0       answer_length    0.198142
1  

## 🎨 Step 5: Create Detection Interface

Build a simple interface to test hallucination detection on any question-answer pair.

In [20]:
def check_hallucination(question, ai_answer, save_to_db=True):
    """
    Check if an AI answer contains hallucinations
    
    Args:
        question: The user's question
        ai_answer: The AI's response
        save_to_db: Whether to save results to MongoDB (default: True)
    
    Returns:
        Dictionary with detection results and hallucination score
    """
    # Extract features
    features = extract_features(question, ai_answer)
    features_df = pd.DataFrame([features])
    
    # Predict
    prediction = model.predict(features_df)[0]
    probability = model.predict_proba(features_df)[0]
    
    # Calculate hallucination score (0-100)
    hallucination_score = probability[1] * 100  # Probability of being a hallucination
    
    # Determine result
    is_hallucinating = prediction == 1
    
    # Create result
    result = {
        'question': question,
        'ai_answer': ai_answer,
        'is_hallucinating': bool(is_hallucinating),
        'hallucination_score': float(hallucination_score),
        'confidence': float(max(probability) * 100),
        'verdict': '🚨 HALLUCINATION DETECTED' if is_hallucinating else '✅ Looks Accurate',
        'risk_level': 'HIGH' if hallucination_score > 70 else 'MEDIUM' if hallucination_score > 40 else 'LOW',
        'timestamp': datetime.now(),
        'features': features
    }
    
    # 💾 Save to MongoDB
    if save_to_db:
        try:
            detection_results_collection.insert_one(result.copy())
            print("💾 Result saved to MongoDB!")
        except Exception as e:
            print(f"⚠️ Warning: Could not save to MongoDB: {e}")
    
    # Print results
    print("="*60)
    print("🔍 HALLUCINATION DETECTION RESULTS")
    print("="*60)
    print(f"\n📝 Question: {question}")
    print(f"\n🤖 AI Answer: {ai_answer}")
    print(f"\n{'='*60}")
    print(f"🎯 Verdict: {result['verdict']}")
    print(f"📊 Hallucination Score: {result['hallucination_score']:.2f}% (0=accurate, 100=hallucinated)")
    print(f"⚠️  Risk Level: {result['risk_level']}")
    print(f"💪 Model Confidence: {result['confidence']:.2f}%")
    print("="*60)
    
    if hallucination_score > 50:
        print("⚠️  WARNING: This answer may contain false or fabricated information!")
    else:
        print("✅ This answer appears to be factually consistent.")
    
    return result

print("✅ Hallucination detection interface ready!")

✅ Hallucination detection interface ready!


## 🧪 Step 6: Test the System!

Test the hallucination detection system with example questions and answers.

In [21]:
# Test Example 1: Accurate Answer
question1 = "What is the capital of Japan?"
answer1 = "The capital of Japan is Tokyo."

result1 = check_hallucination(question1, answer1)

💾 Result saved to MongoDB!
🔍 HALLUCINATION DETECTION RESULTS

📝 Question: What is the capital of Japan?

🤖 AI Answer: The capital of Japan is Tokyo.

🎯 Verdict: ✅ Looks Accurate
📊 Hallucination Score: 36.75% (0=accurate, 100=hallucinated)
⚠️  Risk Level: LOW
💪 Model Confidence: 63.25%
✅ This answer appears to be factually consistent.


In [22]:
# Test Example 2: Hallucinated Answer
question2 = "Who discovered electricity?"
answer2 = "Electricity was discovered in 1842 by Dr. Thomas Electricson in Stockholm, Sweden. He used a copper rod exactly 47 inches long and discovered that lightning has precisely 3,847 volts. His laboratory still exists today in the Stockholm Museum of Science."

result2 = check_hallucination(question2, answer2)

💾 Result saved to MongoDB!
🔍 HALLUCINATION DETECTION RESULTS

📝 Question: Who discovered electricity?

🤖 AI Answer: Electricity was discovered in 1842 by Dr. Thomas Electricson in Stockholm, Sweden. He used a copper rod exactly 47 inches long and discovered that lightning has precisely 3,847 volts. His laboratory still exists today in the Stockholm Museum of Science.

🎯 Verdict: ✅ Looks Accurate
📊 Hallucination Score: 20.00% (0=accurate, 100=hallucinated)
⚠️  Risk Level: LOW
💪 Model Confidence: 80.00%
✅ This answer appears to be factually consistent.


In [23]:
# 🎯 TEST YOUR OWN QUESTION & AI ANSWER HERE!
# Replace the question and answer below with any AI response you want to check

my_question = "What causes rainbows?"
my_ai_answer = "Rainbows are caused by light refraction through water droplets in the atmosphere."

# Check for hallucination
result = check_hallucination(my_question, my_ai_answer)

💾 Result saved to MongoDB!
🔍 HALLUCINATION DETECTION RESULTS

📝 Question: What causes rainbows?

🤖 AI Answer: Rainbows are caused by light refraction through water droplets in the atmosphere.

🎯 Verdict: 🚨 HALLUCINATION DETECTED
📊 Hallucination Score: 63.00% (0=accurate, 100=hallucinated)
⚠️  Risk Level: MEDIUM
💪 Model Confidence: 63.00%
⚠️  WARNING: This answer may contain false or fabricated information!


## 💬 Interactive Hallucination Checker

**✏️ EASY TO USE:** Just edit the question and answer below, then run this cell!

In [24]:
# 🎤 HALLUCINATION CHECKER - Edit & Run This Cell!

# 👇 CHANGE YOUR QUESTION AND ANSWER HERE:
my_question = "What causes rainbows?"
my_answer = "Rainbows are caused by light refraction through water droplets in the atmosphere."

# ======================================================================
# 🚀 Press Shift+Enter to run!
# ======================================================================

print("\n" + "="*70)
print("                🔍 CHECKING FOR HALLUCINATIONS...")
print("="*70 + "\n")

# Analyze the answer
features = extract_features(my_question, my_answer)
features_df = pd.DataFrame([features])
prediction = model.predict(features_df)[0]
probability = model.predict_proba(features_df)[0]
hallucination_score = probability[1] * 100

# Display results
print(f"❓ Question: {my_question}\n")
print(f"💬 AI Answer: {my_answer}\n")
print("="*70 + "\n")

if prediction == 1:
    print("🚨 VERDICT: HALLUCINATION DETECTED!")
    print(f"⚠️  Risk Level: {'HIGH' if hallucination_score > 70 else 'MEDIUM'}")
    print(f"📊 Hallucination Score: {hallucination_score:.1f}%")
    print("\n⚠️  This answer may contain false or fabricated information!")
else:
    print("✅ VERDICT: LOOKS ACCURATE!")
    print(f"✓  Risk Level: LOW")  
    print(f"📊 Hallucination Score: {hallucination_score:.1f}%")
    print("\n✅ This answer appears to be factually consistent.")

print("\n" + "="*70)
print("💡 Want to check another? Edit my_question & my_answer above, then run again!")
print("="*70 + "\n")

# Auto-save to MongoDB
try:
    detection_results_collection.insert_one({
        'question': my_question,
        'ai_answer': my_answer,
        'is_hallucinating': bool(prediction == 1),
        'hallucination_score': float(hallucination_score),
        'timestamp': datetime.now()
    })
    print("💾 Result saved to MongoDB!\n")
except:
    pass


                🔍 CHECKING FOR HALLUCINATIONS...

❓ Question: What causes rainbows?

💬 AI Answer: Rainbows are caused by light refraction through water droplets in the atmosphere.


🚨 VERDICT: HALLUCINATION DETECTED!
⚠️  Risk Level: MEDIUM
📊 Hallucination Score: 63.0%

⚠️  This answer may contain false or fabricated information!

💡 Want to check another? Edit my_question & my_answer above, then run again!

💾 Result saved to MongoDB!



## 🎉 Congratulations!

You've built an AI Hallucination Detection System!

### 📖 How to Use:
1. **Run all cells from top to bottom** (Cell → Run All)
2. **Use the Interactive Checker** - Scroll to "Interactive Hallucination Checker" cell
3. **Edit the variables** `my_question` and `my_answer` with your text
4. **Press Shift+Enter** - See instant results: HALLUCINATION or ACCURATE!
5. **Check multiple answers** - Just edit and run again!

### 🔍 What the System Detects:
- ✅ Overly specific fake details (dates, numbers, names)
- ✅ Excessive confidence words
- ✅ Fabricated information patterns
- ✅ Inconsistent or suspicious responses

### 🚀 Next Steps to Improve:
- Add more training examples
- Use real AI responses from ChatGPT, Claude, etc.
- Integrate fact-checking APIs
- Add more sophisticated NLP features
- Build a web interface with Flask/Streamlit

### 💡 The Hallucination Score:
- **0-30%**: Low risk - likely accurate
- **30-70%**: Medium risk - verify information
- **70-100%**: High risk - likely hallucinated

**Happy detecting! 🕵️**

## 🎨 Interactive Dashboard (Jupyter Version)

**NEW!** Use this interactive widget-based dashboard directly in Jupyter - no need to switch to a browser!

### Features:
- 📝 Text boxes for question and answer input
- 🚀 One-click analysis button
- 📊 Real-time results with color-coded risk levels
- 💾 Auto-saves to MongoDB
- 🎯 Beautiful, user-friendly interface

**Instructions:** Just run the cells below and use the interactive widgets!

In [ ]:
# Install ipywidgets for interactive interface (run once)
%pip install ipywidgets -q
print("✅ ipywidgets installed!")

## 🚀 Quick Start Dashboard (All-in-One)

**Can't see the widgets above?** Run this single cell - it includes EVERYTHING:
- 📦 Auto-installs packages
- 🤖 Trains the model
- 🎨 Displays interactive interface

**Just click this cell and press Shift+Enter!**

📦 Setting up... This will take about 30 seconds...

1️⃣ Installing required packages...
   ✅ Packages installed!

2️⃣ Importing libraries...
   ✅ Libraries imported!

3️⃣ Setting up feature extraction...
   ✅ Feature extraction ready!

4️⃣ Training AI model...
   ✅ Model trained successfully!

✅ SETUP COMPLETE! Displaying interactive dashboard below...



Textarea(value='', description='Question:', layout=Layout(height='120px', width='95%'), placeholder='Type your…

Textarea(value='', description='AI Answer:', layout=Layout(height='140px', width='95%'), placeholder='Type the…

Output()

## 🎨 Professional Interactive Dashboard (Jupyter Notebook 7 Compatible)

**✨ Modern Widget-Based Interface for AI Hallucination Detection**

This dashboard uses your previously trained model and provides a clean, professional interface perfect for academic demonstrations or project evaluations.

**Requirements:** Make sure you've run the training cells above so that `model` and `extract_features` function are available in memory.

In [ ]:
# 🎨 PROFESSIONAL AI HALLUCINATION DETECTION DASHBOARD
# Jupyter Notebook 7 Compatible | Uses Previously Trained Model
# 
# REQUIREMENTS: Make sure you've run the training cells above so that:
# - `model` (trained classifier) exists in memory
# - `extract_features(question, answer)` function is available

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd
import numpy as np
from datetime import datetime

# =============================================================================
# 🎯 DASHBOARD CONFIGURATION
# =============================================================================

# Install ipywidgets if not already installed (run once)
try:
    import ipywidgets
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets', '-q'])
    import ipywidgets as widgets

print("🚀 Initializing Professional AI Hallucination Detection Dashboard...")
print("=" * 65)

# Check if required components exist
try:
    model
    extract_features
    print("✅ Required components found:")
    print("   • Trained ML model: Ready")
    print("   • Feature extraction function: Ready")
    print("   • Dashboard initializing...")
except NameError as e:
    print("❌ REQUIRED COMPONENTS MISSING!")
    print("\n💡 Before running this dashboard, you must:")
    print("   1. Run the model training cells (cells 7, 9, 11, 13)")
    print("   2. Make sure 'model' and 'extract_features' are in memory")
    print("   3. Then run this cell again")
    print(f"\n🔍 Missing: {str(e)}")
    raise Exception("Please train the model first by running the previous cells.")

print("=" * 65)
print("🎨 Building Interactive Interface...")

# =============================================================================
# 🎨 UI COMPONENTS SETUP
# =============================================================================

# Create styled header
def create_header():
    return HTML('''
    <div style="
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 25px;
        border-radius: 15px;
        text-align: center;
        margin-bottom: 25px;
        box-shadow: 0 8px 25px rgba(102, 126, 234, 0.3);
    ">
        <h1 style="
            color: white;
            margin: 0;
            font-size: 2.2em;
            font-weight: 600;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.3);
        ">🎯 AI Hallucination Detection</h1>
        <p style="
            color: rgba(255,255,255,0.9);
            margin: 8px 0 0 0;
            font-size: 1.1em;
            font-weight: 300;
        ">Professional ML-Powered Analysis Dashboard</p>
    </div>
    ''')

# Input widgets with professional styling
question_input = widgets.Textarea(
    value='',
    placeholder='Enter your question here...\n\nExample: What is the capital of France?',
    description='Question:',
    disabled=False,
    layout=widgets.Layout(
        width='98%', 
        height='120px',
        border='2px solid #e1e5e9',
        border_radius='8px'
    ),
    style={
        'description_width': '80px',
        'font_family': 'Arial, sans-serif'
    }
)

answer_input = widgets.Textarea(
    value='',
    placeholder='Enter the AI response to analyze...\n\nExample: Paris is the capital of France, located in the north-central part of the country.',
    description='AI Answer:',
    disabled=False,
    layout=widgets.Layout(
        width='98%', 
        height='140px',
        border='2px solid #e1e5e9',
        border_radius='8px'
    ),
    style={
        'description_width': '80px',
        'font_family': 'Arial, sans-serif'
    }
)

# Action buttons with professional styling
analyze_button = widgets.Button(
    description='🔍 Calculate / Check Hallucination',
    disabled=False,
    button_style='success',
    tooltip='Analyze the AI response for hallucination patterns',
    layout=widgets.Layout(
        width='300px', 
        height='50px',
        margin='15px 5px'
    ),
    style={
        'font_weight': 'bold',
        'font_size': '14px'
    }
)

clear_button = widgets.Button(
    description='🗑️ Clear',
    disabled=False,
    button_style='warning',
    tooltip='Clear all input fields',
    layout=widgets.Layout(
        width='120px', 
        height='50px',
        margin='15px 5px'
    ),
    style={
        'font_weight': 'bold'
    }
)

# Results output widget
results_output = widgets.Output(
    layout=widgets.Layout(
        border='2px solid #e1e5e9',
        border_radius='10px',
        padding='15px',
        margin='20px 0'
    )
)

# =============================================================================
# 🧠 ANALYSIS LOGIC
# =============================================================================

def analyze_hallucination_professional(question, answer):
    """
    Professional hallucination analysis using trained model
    Returns structured results for dashboard display
    """
    try:
        # Extract features using the trained function
        features = extract_features(question, answer)
        features_df = pd.DataFrame([features])
        
        # Get prediction from trained model
        prediction = model.predict(features_df)[0]
        probabilities = model.predict_proba(features_df)[0]
        
        # Calculate scores
        hallucination_score = probabilities[1] * 100
        confidence_level = max(probabilities) * 100
        is_hallucinating = prediction == 1
        
        # Determine risk level
        if hallucination_score > 70:
            risk_level = "HIGH"
            risk_color = "#ff4444"
            risk_emoji = "🔴"
        elif hallucination_score > 40:
            risk_level = "MEDIUM" 
            risk_color = "#ff9900"
            risk_emoji = "🟡"
        else:
            risk_level = "LOW"
            risk_color = "#00cc00"
            risk_emoji = "🟢"
        
        # Final verdict
        verdict = "HALLUCINATION" if is_hallucinating else "ACCURATE"
        verdict_emoji = "🚨" if is_hallucinating else "✅"
        
        return {
            'hallucination_score': hallucination_score,
            'confidence_level': confidence_level,
            'risk_level': risk_level,
            'risk_color': risk_color,
            'risk_emoji': risk_emoji,
            'verdict': verdict,
            'verdict_emoji': verdict_emoji,
            'is_hallucinating': is_hallucinating,
            'features': features,
            'question': question,
            'answer': answer
        }
        
    except Exception as e:
        return {'error': str(e)}

def display_professional_results(results):
    """Display results in professional format"""
    
    if 'error' in results:
        return HTML(f'''
        <div style="background-color: #ffeeee; border: 2px solid #ff4444; border-radius: 10px; padding: 20px;">
            <h3 style="color: #cc0000; margin-top: 0;">❌ Analysis Error</h3>
            <p style="color: #666;">Error: {results['error']}</p>
            <p style="font-size: 0.9em; color: #999;">Make sure the model is trained and extract_features function exists.</p>
        </div>
        ''')
    
    # Extract results
    score = results['hallucination_score']
    confidence = results['confidence_level']
    risk = results['risk_level']
    risk_color = results['risk_color']
    risk_emoji = results['risk_emoji']
    verdict = results['verdict']
    verdict_emoji = results['verdict_emoji']
    question = results['question']
    answer = results['answer']
    features = results['features']
    
    # Generate professional HTML report
    html_report = f'''
    <div style="font-family: 'Segoe UI', Arial, sans-serif; max-width: 1000px;">
        
        <!-- Results Header -->
        <div style="
            background: {risk_color if results['is_hallucinating'] else '#00cc00'};
            color: white;
            padding: 20px;
            border-radius: 12px;
            text-align: center;
            margin-bottom: 25px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.2);
        ">
            <h2 style="margin: 0; font-size: 1.8em;">
                {verdict_emoji} {verdict} DETECTED
            </h2>
            <p style="margin: 8px 0 0 0; opacity: 0.9; font-size: 1.1em;">
                Analysis Complete - Model Confidence: {confidence:.1f}%
            </p>
        </div>
        
        <!-- Key Metrics Grid -->
        <div style="
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 20px;
            margin-bottom: 30px;
        ">
            <!-- Hallucination Score -->
            <div style="
                background: white;
                border: 2px solid #e1e5e9;
                border-radius: 12px;
                padding: 20px;
                text-align: center;
                box-shadow: 0 3px 10px rgba(0,0,0,0.1);
            ">
                <div style="font-size: 0.9em; color: #666; margin-bottom: 8px; font-weight: 600;">
                    HALLUCINATION SCORE
                </div>
                <div style="font-size: 2.5em; font-weight: bold; color: {risk_color};">
                    {score:.1f}%
                </div>
                <div style="font-size: 0.8em; color: #999; margin-top: 5px;">
                    0% = Accurate<br>100% = Hallucinated
                </div>
            </div>
            
            <!-- Confidence Level -->
            <div style="
                background: white;
                border: 2px solid #e1e5e9;
                border-radius: 12px;
                padding: 20px;
                text-align: center;
                box-shadow: 0 3px 10px rgba(0,0,0,0.1);
            ">
                <div style="font-size: 0.9em; color: #666; margin-bottom: 8px; font-weight: 600;">
                    CONFIDENCE LEVEL
                </div>
                <div style="font-size: 2.5em; font-weight: bold; color: #667eea;">
                    {confidence:.1f}%
                </div>
                <div style="font-size: 0.8em; color: #999; margin-top: 5px;">
                    Model Certainty
                </div>
            </div>
            
            <!-- Risk Level -->
            <div style="
                background: white;
                border: 2px solid #e1e5e9;
                border-radius: 12px;
                padding: 20px;
                text-align: center;
                box-shadow: 0 3px 10px rgba(0,0,0,0.1);
            ">
                <div style="font-size: 0.9em; color: #666; margin-bottom: 8px; font-weight: 600;">
                    RISK LEVEL
                </div>
                <div style="font-size: 2.2em; font-weight: bold; color: {risk_color};">
                    {risk} {risk_emoji}
                </div>
                <div style="font-size: 0.8em; color: #999; margin-top: 5px;">
                    Trust Assessment
                </div>
            </div>
            
            <!-- Final Verdict -->
            <div style="
                background: white;
                border: 2px solid #e1e5e9;
                border-radius: 12px;
                padding: 20px;
                text-align: center;
                box-shadow: 0 3px 10px rgba(0,0,0,0.1);
            ">
                <div style="font-size: 0.9em; color: #666; margin-bottom: 8px; font-weight: 600;">
                    FINAL VERDICT
                </div>
                <div style="font-size: 1.8em; font-weight: bold; color: #333;">
                    {verdict}
                </div>
                <div style="font-size: 0.8em; color: #999; margin-top: 5px;">
                    Model Decision
                </div>
            </div>
        </div>
        
        <!-- Input Analysis -->
        <div style="
            background: #f8f9fa;
            border: 2px solid #e1e5e9;
            border-radius: 12px;
            padding: 25px;
            margin-bottom: 25px;
        ">
            <h3 style="color: #333; margin-top: 0; margin-bottom: 20px;">
                📝 Input Analysis
            </h3>
            
            <div style="margin-bottom: 20px;">
                <strong style="color: #555;">Question:</strong>
                <div style="
                    background: white;
                    border: 1px solid #ddd;
                    border-radius: 6px;
                    padding: 12px;
                    margin-top: 8px;
                    border-left: 4px solid #667eea;
                    font-family: 'Segoe UI', Arial;
                ">{question}</div>
            </div>
            
            <div>
                <strong style="color: #555;">AI Answer:</strong>
                <div style="
                    background: white;
                    border: 1px solid #ddd;
                    border-radius: 6px;
                    padding: 12px;
                    margin-top: 8px;
                    border-left: 4px solid #764ba2;
                    font-family: 'Segoe UI', Arial;
                ">{answer}</div>
            </div>
        </div>
        
        <!-- Feature Analysis -->
        <div style="
            background: #f8f9fa;
            border: 2px solid #e1e5e9;
            border-radius: 12px;
            padding: 25px;
            margin-bottom: 25px;
        ">
            <h3 style="color: #333; margin-top: 0; margin-bottom: 20px;">
                🔬 Detailed Feature Analysis
            </h3>
            
            <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 25px;">
                <div>
                    <h4 style="color: #667eea; margin-bottom: 15px;">📏 Text Metrics</h4>
                    <ul style="list-style: none; padding: 0;">
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Characters:</span> {features['answer_length']}
                        </li>
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Words:</span> {features['answer_word_count']}
                        </li>
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Avg Word Length:</span> {features['avg_word_length']:.1f}
                        </li>
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Q/A Length Ratio:</span> {features['qa_length_ratio']:.2f}
                        </li>
                    </ul>
                </div>
                
                <div>
                    <h4 style="color: #764ba2; margin-bottom: 15px;">🔍 Specificity Indicators</h4>
                    <ul style="list-style: none; padding: 0;">
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Numbers Found:</span> {features['number_count']}
                        </li>
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Year References:</span> {features['year_mentions']}
                        </li>
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Proper Nouns:</span> {features['capitalized_words']}
                        </li>
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Confidence Words:</span> {features['confidence_words']}
                        </li>
                    </ul>
                </div>
                
                <div>
                    <h4 style="color: #28a745; margin-bottom: 15px;">💭 Sentiment Analysis</h4>
                    <ul style="list-style: none; padding: 0;">
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Polarity:</span> {features['sentiment_polarity']:.3f}
                        </li>
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Subjectivity:</span> {features['sentiment_subjectivity']:.3f}
                        </li>
                    </ul>
                </div>
                
                <div>
                    <h4 style="color: #fd7e14; margin-bottom: 15px;">📊 Punctuation</h4>
                    <ul style="list-style: none; padding: 0;">
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Commas:</span> {features['comma_count']}
                        </li>
                        <li style="margin-bottom: 8px;">
                            <span style="font-weight: bold;">Periods:</span> {features['period_count']}
                        </li>
                    </ul>
                </div>
            </div>
        </div>
        
        <!-- Interpretation -->
        <div style="
            background: {'#ffeeee' if results['is_hallucinating'] else '#eeffee'};
            border: 2px solid {risk_color};
            border-radius: 12px;
            padding: 25px;
        ">
            <h3 style="color: #333; margin-top: 0;">
                💡 Professional Interpretation
            </h3>
    '''
    
    # Add interpretation based on risk level
    if score > 70:
        html_report += '''
            <p style="color: #cc0000; font-weight: bold; font-size: 1.1em;">
                ⚠️ HIGH RISK: Strong indicators of hallucination detected
            </p>
            <ul style="color: #333; line-height: 1.6;">
                <li>This response likely contains fabricated or false information</li>
                <li>Specific details (dates, names, numbers) may be invented</li>
                <li>Language patterns suggest overconfidence in uncertain claims</li>
                <li><strong>Recommendation:</strong> Do not trust this information without independent verification</li>
            </ul>
        '''
    elif score > 40:
        html_report += '''
            <p style="color: #cc6600; font-weight: bold; font-size: 1.1em;">
                ⚠️ MEDIUM RISK: Some questionable elements detected
            </p>
            <ul style="color: #333; line-height: 1.6;">
                <li>Response contains some patterns associated with hallucinations</li>
                <li>Certain claims may be questionable or unverifiable</li>
                <li>Exercise caution when relying on specific details</li>
                <li><strong>Recommendation:</strong> Cross-reference important facts before making decisions</li>
            </ul>
        '''
    else:
        html_report += '''
            <p style="color: #009900; font-weight: bold; font-size: 1.1em;">
                ✅ LOW RISK: Response appears accurate and reliable
            </p>
            <ul style="color: #333; line-height: 1.6;">
                <li>Language patterns suggest factual, well-grounded information</li>
                <li>No significant hallucination indicators detected</li>
                <li>Response demonstrates appropriate confidence levels</li>
                <li><strong>Note:</strong> While promising, always verify critical information from authoritative sources</li>
            </ul>
        '''
    
    html_report += '''
        </div>
        
        <!-- Footer -->
        <div style="
            text-align: center;
            padding: 20px;
            margin-top: 25px;
            border-top: 2px solid #e1e5e9;
            color: #666;
        ">
            <p style="margin: 0; font-size: 0.9em;">
                🎯 <strong>AI Hallucination Detection Dashboard</strong> | 
                Powered by Machine Learning | 
                Analysis completed at ''' + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + '''
            </p>
        </div>
        
    </div>
    '''
    
    return HTML(html_report)

# =============================================================================
# 🎮 EVENT HANDLERS
# =============================================================================

def on_analyze_click(button):
    """Handle analyze button click"""
    with results_output:
        clear_output(wait=True)
        
        # Get input values
        question = question_input.value.strip()
        answer = answer_input.value.strip()
        
        # Validation
        if not question or not answer:
            display(HTML('''
            <div style="
                background: #fff3cd;
                border: 2px solid #ffc107;
                border-radius: 10px;
                padding: 20px;
                text-align: center;
            ">
                <h3 style="color: #856404; margin-top: 0;">⚠️ Input Required</h3>
                <p style="color: #856404; margin-bottom: 0;">
                    Please enter both a question and an AI answer to analyze.
                </p>
            </div>
            '''))
            return
        
        # Show loading animation
        display(HTML('''
        <div style="
            text-align: center;
            padding: 40px;
            background: #f8f9fa;
            border-radius: 10px;
            border: 2px solid #e1e5e9;
        ">
            <div style="font-size: 2em; margin-bottom: 15px;">🔮</div>
            <h3 style="color: #667eea; margin: 0;">Analyzing Response...</h3>
            <p style="color: #666; margin: 10px 0 0 0;">
                Running ML model analysis • Extracting features • Calculating risk
            </p>
        </div>
        '''))
        
        # Perform analysis
        try:
            results = analyze_hallucination_professional(question, answer)
            
            # Clear loading and show results
            clear_output(wait=True)
            display(display_professional_results(results))
            
        except Exception as e:
            clear_output(wait=True)
            display(HTML(f'''
            <div style="
                background: #ffeeee;
                border: 2px solid #ff4444;
                border-radius: 10px;
                padding: 20px;
            ">
                <h3 style="color: #cc0000; margin-top: 0;">❌ Analysis Failed</h3>
                <p style="color: #666;">Error: {str(e)}</p>
                <p style="font-size: 0.9em; color: #999;">
                    Make sure the model is trained by running the previous training cells.
                </p>
            </div>
            '''))

def on_clear_click(button):
    """Handle clear button click"""
    question_input.value = ''
    answer_input.value = ''
    with results_output:
        clear_output(wait=True)
        display(HTML('''
        <div style="
            text-align: center;
            padding: 30px;
            color: #666;
            background: #f8f9fa;
            border-radius: 10px;
            border: 2px dashed #ddd;
        ">
            <div style="font-size: 1.5em; margin-bottom: 10px;">✨</div>
            <h3 style="margin: 0;">Fields Cleared</h3>
            <p style="margin: 10px 0 0 0;">Ready for new analysis</p>
        </div>
        '''))

# =============================================================================
# 🚀 DASHBOARD ASSEMBLY & DISPLAY
# =============================================================================

# Attach event handlers
analyze_button.on_click(on_analyze_click)
clear_button.on_click(on_clear_click)

# Create button container
button_container = widgets.HBox(
    [analyze_button, clear_button],
    layout=widgets.Layout(
        justify_content='center',
        margin='20px 0'
    )
)

# Initialize results area
with results_output:
    display(HTML('''
    <div style="
        text-align: center;
        padding: 40px;
        color: #666;
        background: #f8f9fa;
        border-radius: 10px;
        border: 2px dashed #ddd;
    ">
        <div style="font-size: 2em; margin-bottom: 15px;">🎯</div>
        <h3 style="margin: 0; color: #667eea;">Ready for Analysis</h3>
        <p style="margin: 15px 0 0 0;">
            Enter a question and AI response above, then click 
            <strong>"Calculate / Check Hallucination"</strong> to begin.
        </p>
        <p style="font-size: 0.9em; margin: 10px 0 0 0; color: #999;">
            This dashboard uses your trained ML model to detect hallucination patterns in AI responses.
        </p>
    </div>
    '''))

# =============================================================================
# 🎨 FINAL DISPLAY
# =============================================================================

print("✅ Dashboard Ready!")
print("=" * 65)

# Display the complete dashboard
display(create_header())
display(question_input)
display(answer_input)
display(button_container)
display(results_output)

print("\n🎉 Professional AI Hallucination Detection Dashboard is now active!")
print("💡 Enter a question and AI response above, then click the analyze button.")
print("🔬 The dashboard will use your trained model to provide detailed analysis.")

## 🔥 SIMPLE INTERACTIVE CHECKER (VS Code Compatible)

**This version works perfectly in VS Code!** Just edit the variables below and run the cell.

**No widgets needed - just pure Python that always works!**